In [12]:
import pandas as pd
import re

df = pd.read_excel('recruitment_agency/microsoft_outlook_leads.xlsx')

def split_emails(row):
    """Split rows with multiple emails separated by common delimiters"""
    email_col = 'Email'
    phone_col = 'Phone'  # Adjust if your column name is different
    
    if pd.isna(row[email_col]):
        return [row]
    
    # Split by common delimiters: comma, semicolon, space, pipe
    emails = re.split(r'[,;\s|]+', str(row[email_col]))
    emails = [e.strip() for e in emails if e.strip()]
    
    if len(emails) <= 1:
        return [row]
    
    # Create duplicate rows for each email
    rows = []
    for i, email in enumerate(emails):
        new_row = row.copy()
        new_row[email_col] = email
        # Keep phone number only for the first row, clear it for duplicates
        if i > 0:
            new_row[phone_col] = None  # or '' for empty string
        rows.append(new_row)
    return rows

# Apply the splitting function
expanded_rows = []
for _, row in df.iterrows():
    expanded_rows.extend(split_emails(row))

df = pd.DataFrame(expanded_rows).reset_index(drop=True)

def extract_business_name(name):
    """Extract the first part of business name before - or |"""
    if pd.isna(name):
        return name
    name = str(name)
    # Split by - or | and take the first part
    parts = re.split(r'\s*[-|:–,]\s*|\s+I\s+', name)
    return parts[0].strip()

df['Name'] = df['Name'].apply(extract_business_name)

def is_sentry_email(email):
    """Check if email matches sentry pattern"""
    if pd.isna(email):
        return False
    email = str(email).lower()
    # Pattern: 32 hex characters @ sentry domains
    pattern = r'^[a-f0-9]{32}@sentry.*\.(wixpress\.com|io)$'
    return bool(re.match(pattern, email))

# Filter out sentry emails
df = df[~df['Email'].apply(is_sentry_email)].reset_index(drop=True)

print(f"Rows before removing duplicates: {len(df)}")
df = df.drop_duplicates(subset=['Name', 'Email'], keep='first').reset_index(drop=True)
print(f"Rows after removing duplicates: {len(df)}")

# Display results
print(f"\nTotal rows after processing: {len(df)}")
print("\nFirst few rows:")
print(df.head(10))

# Save to new Excel file
df.to_excel('recruitment_agency/microsoft_outlook_cleaned_leads.xlsx', index=False)
print("\nData saved to 'cleaned_data.xlsx'")

Rows before removing duplicates: 118
Rows after removing duplicates: 90

Total rows after processing: 90

First few rows:
                                                 URL            Industry  \
0  https://www.google.com/maps/place/Overseas+Pla...  Recruitment Agency   
1  https://www.google.com/maps/place/Recruise+Ind...  Recruitment Agency   
2  https://www.google.com/maps/place/GLOBE+CONSUL...  Recruitment Agency   
3  https://www.google.com/maps/place/Futurz+Consu...  Recruitment Agency   
4  https://www.google.com/maps/place/Inspiration+...  Recruitment Agency   
5  https://www.google.com/maps/place/AJEETS/data=...  Recruitment Agency   
6  https://www.google.com/maps/place/AJEETS/data=...  Recruitment Agency   
7  https://www.google.com/maps/place/AJEETS/data=...  Recruitment Agency   
8  https://www.google.com/maps/place/BSS+Recruit/...  Recruitment Agency   
9  https://www.google.com/maps/place/Falcon+Servi...  Recruitment Agency   

                                Name     